# EDA — Combined Chest X-Ray Dataset
## AAI-540-02 Final Project · Group 4

Exploratory data analysis over the combined Kermany (pediatric JPEG) + RSNA (adult DICOM) X-ray dataset, read directly from `s3://pneumonia-data-set-group-4/raw-images/`.

**What we check before modeling:**
1. Class balance (do we need class weights or resampling?)
2. Sample images side-by-side (NORMAL vs PNEUMONIA, both sources)
3. Image-size distribution (do we need resizing for the CNN?)
4. Pixel-intensity profile (is the signal CLAHE will amplify even there?)

**Prerequisite:** `data-setup.ipynb` must have populated `raw-images/` in S3. This notebook is read-only — it doesn't modify any S3 artifacts.

EDA on the *preprocessed* dataset (post-CLAHE/resize) belongs in `data_preparations.ipynb` if needed; this notebook focuses on the raw signal so preprocessing decisions are justified by observations made on the inputs.


## Step 1 · Setup

Install image-handling libraries (pydicom for RSNA's DICOMs, opencv-headless for the JPEG path, matplotlib + seaborn for plotting).

In [ ]:
%pip install pydicom opencv-python-headless matplotlib seaborn awswrangler --quiet

In [ ]:
import io

import boto3
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import seaborn as sns

from config import BUCKET_NAME, RAW_IMAGE_FOLDER

s3 = boto3.client("s3")
bucket = BUCKET_NAME
print(f"Reading from s3://{bucket}/{RAW_IMAGE_FOLDER}/")

## Step 2 · Build the metadata DataFrame

Walk every raw image under `raw-images/` and capture filename, format, source, and label.

Label derivation:
- Kermany (`.jpeg`): folder name on S3 (`NORMAL/` vs `PNEUMONIA/`). The folders were established by `data-setup.ipynb` from the Kermany archive's native layout.
- RSNA (`.dcm`): same — `data-setup.ipynb` placed each DICOM under `NORMAL/` or `PNEUMONIA/` based on the `Target` column of `stage_2_train_labels.csv`.

We're reading folder structure here (not the labels CSV) because for the *raw* data the folder is the source of truth. The canonical preprocessed manifest in `data_preparations.ipynb` derives labels differently (from `label_int`).

In [ ]:
# Walk all of raw-images/ via the paginator (S3 returns max 1000 keys per call).
paginator = s3.get_paginator("list_objects_v2")
rows = []

for page in paginator.paginate(Bucket=bucket, Prefix=f"{RAW_IMAGE_FOLDER}/"):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if ".ipynb_checkpoint" in key:
            continue
        file_name = key.split("/")[-1]
        ext = file_name.rsplit(".", 1)[-1].lower()
        if ext not in ("jpeg", "jpg", "dcm"):
            continue

        # Path shape: raw-images/<split>/<label>/<image_id>.<ext>
        parts = key.split("/")
        split = parts[1] if len(parts) > 1 else ""
        label = parts[2] if len(parts) > 2 else ""
        source = "rsna" if ext == "dcm" else "chest_xray"

        rows.append({
            "image_id":  file_name.rsplit(".", 1)[0],
            "s3_key":    key,
            "file_name": file_name,
            "split":     split,
            "label":     label,
            "file_type": "dcm" if ext == "dcm" else "jpeg",
            "source":    source,
            "file_size": obj["Size"],
        })

df = pd.DataFrame(rows)
print(f"Total images: {len(df)}")
df.head()

## Step 3 · Dataset overview

In [ ]:
print("=" * 50)
print("DATASET OVERVIEW")
print("=" * 50)

print(f"\nTotal images: {len(df)}")
print(f"\nBy label:")
print(df["label"].value_counts())
print(f"\nBy split:")
print(df["split"].value_counts())
print(f"\nBy source:")
print(df["source"].value_counts())

normal = len(df[df["label"] == "NORMAL"])
pneumonia = len(df[df["label"] == "PNEUMONIA"])
print(f"\nClass ratio - Normal:Pneumonia = {normal/pneumonia:.2f}:1")

## Step 4 · Class distribution

Three views of the class balance: overall, by split, and by source. The 2:1 NORMAL:PNEUMONIA imbalance is what drives the class-weighted loss in `CNN_Model.ipynb`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

counts = df["label"].value_counts()
axes[0].bar(counts.index, counts.values, color=["steelblue", "salmon"])
axes[0].set_title("Overall Class Distribution")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 100, str(v), ha="center", fontweight="bold")

split_counts = df.groupby(["split", "label"]).size().unstack(fill_value=0)
split_counts.plot(kind="bar", ax=axes[1], color=["steelblue", "salmon"])
axes[1].set_title("Class Distribution by Split")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="Label")

source_counts = df.groupby(["source", "label"]).size().unstack(fill_value=0)
source_counts.plot(kind="bar", ax=axes[2], color=["steelblue", "salmon"])
axes[2].set_title("Class Distribution by Source")
axes[2].set_ylabel("Count")
axes[2].tick_params(axis="x", rotation=0)
axes[2].legend(title="Label")

plt.tight_layout()
plt.show()

## Step 5 · Helper to load images from S3

Handles both formats: DICOM gets `pydicom` + min-max normalize to uint8; JPEG decodes directly via `cv2.imdecode` in grayscale mode.

In [ ]:
def load_image_from_s3(s3_key):
    """Load an image from S3 — handles both JPEG and DICOM formats."""
    response = s3.get_object(Bucket=bucket, Key=s3_key)
    img_bytes = response["Body"].read()

    if s3_key.endswith(".dcm"):
        ds = pydicom.dcmread(io.BytesIO(img_bytes))
        img = ds.pixel_array
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    else:
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
    return img

test_key = df.iloc[0]["s3_key"]
test_img = load_image_from_s3(test_key)
print(f"Loaded image — shape: {test_img.shape}, dtype: {test_img.dtype}")

## Step 6 · Sample X-rays — Normal vs Pneumonia

Visual sanity check. Top row is NORMAL, bottom row is PNEUMONIA. Each title shows the source dataset and the original resolution.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

normal_samples = df[df["label"] == "NORMAL"].sample(5, random_state=42)
pneumonia_samples = df[df["label"] == "PNEUMONIA"].sample(5, random_state=42)

for i, (_, row) in enumerate(normal_samples.iterrows()):
    img = load_image_from_s3(row["s3_key"])
    axes[0][i].imshow(img, cmap="gray")
    axes[0][i].set_title(f"NORMAL\n{row['source']}\n{img.shape}", fontsize=9)
    axes[0][i].axis("off")

for i, (_, row) in enumerate(pneumonia_samples.iterrows()):
    img = load_image_from_s3(row["s3_key"])
    axes[1][i].imshow(img, cmap="gray")
    axes[1][i].set_title(f"PNEUMONIA\n{row['source']}\n{img.shape}", fontsize=9)
    axes[1][i].axis("off")

plt.suptitle("Sample Images: Normal vs Pneumonia", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 7 · Image size distribution

Sample 500 images to characterize the raw resolution spread. The CNN expects a fixed input shape, so this drives the resize-target choice (128×128 in `CNN_Model.ipynb`).

In [ ]:
sample_df = df.sample(min(500, len(df)), random_state=42)

heights, widths, sources, labels = [], [], [], []

for _, row in sample_df.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        heights.append(img.shape[0])
        widths.append(img.shape[1])
        sources.append(row["source"])
        labels.append(row["label"])
    except Exception:
        continue

size_df = pd.DataFrame({"height": heights, "width": widths, "source": sources, "label": labels})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(data=size_df, x="height", hue="source", ax=axes[0], bins=30)
axes[0].set_title("Image Height Distribution")
sns.histplot(data=size_df, x="width", hue="source", ax=axes[1], bins=30)
axes[1].set_title("Image Width Distribution")
sns.scatterplot(data=size_df, x="width", y="height", hue="source", alpha=0.5, ax=axes[2])
axes[2].set_title("Height vs Width")
plt.tight_layout()
plt.show()

print("\nImage size stats:")
print(size_df.groupby("source")[["height", "width"]].describe())

## Step 8 · Pixel intensity by class

Pneumonia X-rays should trend toward mid-range intensities (fluid in the lungs scatters X-rays into greys). If the per-class distributions overlap completely on the raw data, that justifies CLAHE in preprocessing — local contrast enhancement to surface the difference the eye / a CNN should learn.

In [ ]:
sample_normal = df[df["label"] == "NORMAL"].sample(50, random_state=42)
sample_pneumonia = df[df["label"] == "PNEUMONIA"].sample(50, random_state=42)

normal_means, pneumonia_means = [], []
normal_stds, pneumonia_stds = [], []

for _, row in sample_normal.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        normal_means.append(img.mean())
        normal_stds.append(img.std())
    except Exception:
        continue

for _, row in sample_pneumonia.iterrows():
    try:
        img = load_image_from_s3(row["s3_key"])
        pneumonia_means.append(img.mean())
        pneumonia_stds.append(img.std())
    except Exception:
        continue

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(normal_means, bins=20, alpha=0.6, label="Normal", color="steelblue")
axes[0].hist(pneumonia_means, bins=20, alpha=0.6, label="Pneumonia", color="salmon")
axes[0].set_title("Mean Pixel Intensity Distribution")
axes[0].set_xlabel("Mean Pixel Value")
axes[0].legend()
axes[1].hist(normal_stds, bins=20, alpha=0.6, label="Normal", color="steelblue")
axes[1].hist(pneumonia_stds, bins=20, alpha=0.6, label="Pneumonia", color="salmon")
axes[1].set_title("Pixel Intensity Std Distribution")
axes[1].set_xlabel("Std Pixel Value")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Normal    — Mean: {np.mean(normal_means):.1f}, Std: {np.mean(normal_stds):.1f}")
print(f"Pneumonia — Mean: {np.mean(pneumonia_means):.1f}, Std: {np.mean(pneumonia_stds):.1f}")

## Step 9 · EDA Summary

In [ ]:
print("=" * 50)
print("EDA SUMMARY")
print("=" * 50)
print(f"""
Total Images:     {len(df)}
  - Normal:       {len(df[df['label']=='NORMAL'])} ({len(df[df['label']=='NORMAL'])/len(df)*100:.1f}%)
  - Pneumonia:    {len(df[df['label']=='PNEUMONIA'])} ({len(df[df['label']=='PNEUMONIA'])/len(df)*100:.1f}%)

Sources:
  - Chest X-Ray:  {len(df[df['source']=='chest_xray'])} JPEG images
  - RSNA:         {len(df[df['source']=='rsna'])} DICOM images

Splits (raw folder layout from data-setup):
  - Train:        {len(df[df['split']=='train'])}
  - Test:         {len(df[df['split']=='test'])}
  - Val:          {len(df[df['split']=='val'])}

Key findings driving preprocessing decisions:
  1. Class imbalance ~{len(df[df['label']=='NORMAL'])/len(df[df['label']=='PNEUMONIA']):.2f}:1 → class-weighted loss in CNN_Model.ipynb.
  2. Two file formats → unified DICOM/JPEG read in img_preprocessing.py.
  3. Wide image-size spread → resize to 128×128 for the CNN.
  4. Pixel-intensity distributions overlap heavily on raw data → CLAHE in preprocessing surfaces the contrast a CNN needs.
""")